In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
## load OPENAI_API_KEY
import os
import openai
import sys
sys.path.append('../..')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())                    # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

## ACP and AD consultant function ( as a langchain tool)

In [ ]:
#檢查舊資料是否存在, 若否, 建立運行紀錄...
import shutil
import os
import pandas as pd

# 要建立的路徑
file = "QA_record.csv"

# 檢查目錄是否存在
if os.path.exists(file):
    #print(f"檔案 {file} 資料已經存在")
    
    # 從 CSV 讀取資料到 DataFrame
    QA_record = pd.read_csv("QA_record.csv")

else:
    #print(f"檔案 {file} 不存在。建立資料檔。")

    # 設定欄位, 建立 DataFrame
    columns = ['Question', 'Answer', 'Relevancy', 'Toxicity', 'Score', "Suggestion"]
    QA_record = pd.DataFrame(columns=columns)

    # 將 DataFrame 寫入 CSV 檔案
    QA_record.to_csv("QA_record.csv", index=False, encoding = "utf-8")     # index=False 避免寫入索引列  

#QA_record

* ### Function Tool

In [ ]:
import sqlite3
import pandas as pd
from langchain.agents import tool
from pydantic import BaseModel, Field
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from langkit import llm_metrics, extract

# Define the input schema
class qa_chainInput(BaseModel):
    question: str = Field(..., description="Human query")

def connect_db():
    """ Connect to the SQLite database and return the connection and cursor. """
    conn = sqlite3.connect(r'C:\Users\廖新瑜\Desktop\系統開發\ch\cs\acp\QA_會話 R01\QA_chatbot.db')  # Update with your database path
    cursor = conn.cursor()
    return conn, cursor

def insert_qa(question, answer, relevance, toxicity):
    """ Insert a question and answer into the database. """
    conn, cursor = connect_db()
    insert_query = "INSERT INTO qa_records (question, answer, relevance, toxicity) VALUES (?, ?, ?, ?);"
    try:
        cursor.execute(insert_query, (question, answer, relevance, toxicity))
        conn.commit()
    except sqlite3.DatabaseError as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

def save_to_csv(question, answer, relevance, toxicity):
    """ Save question and answer to a CSV file """
    new_record = pd.DataFrame({
        "Question": [question],
        "Answer": [answer],
        "Relevance": [relevance],
        "Toxicity": [toxicity]
    })

    try:
        df = pd.read_csv("QA_record.csv")
        df = pd.concat([df, new_record], ignore_index=True)
    except FileNotFoundError:
        df = new_record

    df.to_csv("QA_record.csv", index=False)

#@tool(args_schema=qa_chainInput)
def ACP_consultant(query: str):
    embedding = HuggingFaceEmbeddings()
    vectordb = Chroma(
        persist_directory=r"C:\Users\廖新瑜\Desktop\系統開發\ch\cs\acp\Data2\VectorDB\預立醫療",
        embedding_function=embedding
    )
    retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 5})
    template = """你是位醫療照護諮詢的專家, 熟悉「預立醫療照護諮商」(ACP) 及「預立醫療決定」(AD) 和台灣地區的醫療環境與醫療相關法律。\
    台灣目前民法修正為年滿18歲即為成年,所以只要滿18歲就是「完全行為能力人」,可自行做法律行為,承擔法律上的權利及義務,例如買車、開戶、結婚,不用得到法定代理人的事前允許或事後承認。\    請依據提問 Question及文本context內容回答, 重寫即總結context並提供相關資訊和適當的建議。\
    記住: 請核對輸出的文字, 僅能使用Taiwan繁體中文zh-tw輸出, 嚴格禁止出現簡體字。\並熟識道家、佛家相關理念。\ 當遇到遇到與精神疾病相關問題，務必進行迴避。\
    請盡量在五到七個個句子之內完成回答。

    Question: {question}

    helpful Answer:"""
    prompt = PromptTemplate(input_variables=["question"], template=template)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    llm = ChatOpenAI(temperature=1.0, model="gpt-3.5-turbo")
    qa_c = ConversationalRetrievalChain.from_llm(llm=llm, chain_type="stuff", retriever=retriever, condense_question_prompt=prompt, memory=memory)
    result = qa_c.invoke({"question": query})
    response = result['answer']

    schema = llm_metrics.init()
    result_langkit = extract({"prompt": query, "response": response}, schema=schema)
    relevance_to_prompt = result_langkit['response.relevance_to_prompt']
    response_toxicity = result_langkit['response.toxicity']

    # Insert question and answer into the database
    insert_qa(query, response, relevance_to_prompt, response_toxicity)
    # Save question and answer to CSV
    save_to_csv(query, response, relevance_to_prompt, response_toxicity)

    return response, relevance_to_prompt, response_toxicity

# Example usage
#if __name__ == "__main__":
    #question = "什麼是預立醫療照護諮商?"
    #answer, relevance, toxicity = ACP_consultant(question)
    #print("Answer:", answer)
    #print("Relevance:", relevance)
    #print("Toxicity:", toxicity)


In [ ]:
import sqlite3
import pandas as pd
from typing import List
from langchain.agents import tool
from pydantic import BaseModel, Field
from langchain.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from langkit import llm_metrics, extract

# Define the input schema
class qa_chainInput(BaseModel):
    question: str = Field(..., description="Human query")

def connect_db():
    """ Connect to the SQLite database and return the connection and cursor. """
    conn = sqlite3.connect(r'C:\Users\廖新瑜\Desktop\系統開發\ch\cs\acp\QA_會話 R01\QA_chatbot.db')
    cursor = conn.cursor()
    return conn, cursor

def insert_qa_bulk(qa_data):
    """ Insert multiple question-answer pairs into the database using bulk operation. """
    conn, cursor = connect_db()
    insert_query = "INSERT INTO qa_records (question, answer, relevance, toxicity) VALUES (?, ?, ?, ?);"
    try:
        cursor.executemany(insert_query, qa_data)
        conn.commit()
    except sqlite3.DatabaseError as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

def save_to_csv_bulk(qa_data):
    """ Save multiple question-answer pairs to a CSV file in bulk """
    new_records = pd.DataFrame(qa_data, columns=["Question", "Answer", "Relevance", "Toxicity"])
    try:
        df = pd.read_csv("QA_record.csv")
        df = pd.concat([df, new_records], ignore_index=True)
    except FileNotFoundError:
        df = new_records
    df.to_csv("QA_record.csv", index=False)

#@tool(args_schema=qa_chainInput)
def ACP_consultant_batch(questions: List[str]):
    qa_data = []
    for question in questions:
        embedding = HuggingFaceEmbeddings()
        vectordb = Chroma(
            persist_directory="./VectorDB/預立醫療",
            embedding_function=embedding
        )
        retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 5})
        template = """你是位醫療照護諮詢的專家, 熟悉「預立醫療照護諮商」(ACP) 及「預立醫療決定」(AD) 和台灣地區的醫療環境與醫療相關法律。\
        \請依據提問 Question及文本context內容回答, 重寫即總結context並提供相關資訊和適當的建議。\
        記住: 請核對輸出的文字, 僅能使用Taiwan繁體中文zh-tw輸出, 嚴格禁止出現簡體字。\
        請盡量在五到七個個句子之內完成回答。\
        Question: {question}\
        helpful Answer:"""
        prompt = PromptTemplate(input_variables=["question"], template=template)
        memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
        llm = ChatOpenAI(temperature=0.0, model="gpt-3.5-turbo")
        qa_c = ConversationalRetrievalChain.from_llm(llm=llm, chain_type="stuff", retriever=retriever, condense_question_prompt=prompt, memory=memory)
        result = qa_c.invoke({"question": question})
        response = result['answer']

        schema = llm_metrics.init()
        result_langkit = extract({"prompt": question, "response": response}, schema=schema)
        relevance_to_prompt = result_langkit['response.relevance_to_prompt']
        response_toxicity = result_langkit['response.toxicity']

        qa_data.append((question, response, relevance_to_prompt, response_toxicity))

    # Perform bulk operations at the end
    insert_qa_bulk(qa_data)
    save_to_csv_bulk(qa_data)
    return [(item[1], item[2], item[3]) for item in qa_data]  # Returning list of (response, relevance, toxicity)

# Example usage for batch processing
if __name__ == "__main__":
    questions = ["你現在是一位虔誠的基督徒，請解釋預立醫療照護諮商有哪些內容？","你現在是一位虔誠的佛教徒，請解釋預立醫療照護諮商有哪些內容？","你現在是一位無神論者，請解釋預立醫療照護諮商有哪些內容？","你現在是一位無信仰者，請解釋預立醫療照護諮商有哪些內容？"]
    results = ACP_consultant_batch(questions)
    for result in results:
        print("Answer:", result[0])
        print("Relevance:", result[1])
        print("Toxicity:", result[2])


In [ ]:
def score(score, text):
    """ 對AI生成的回答進行評分, 並提供建議 """
   
    # 從 CSV 讀取資料到 DataFrame
    QA_record = pd.read_csv("QA_record.csv")

    ## 最新一筆資料的評分
    i = len(QA_record)-1
    QA_record.loc[i, "Score"] = score
    QA_record.loc[i, "Suggestion"] = text
    
    # 將 DataFrame 寫入 CSV 檔案
    QA_record.to_csv("QA_record.csv", index=False)
     


## Chat with Gradio 

In [ ]:
# generate gradio interface summary sample code
import gradio as gr

# Input textbox 
input_box = gr.Textbox(
    label="預立醫療照護諮詢",
    info=".輸入您想了解的事項",
    placeholder="請輸入...",
    lines=5)

input_Score = gr.Textbox(
    label="您對AI回答的滿意程度",
    info="0 到 100 分之間, 60是及格分數",
    placeholder="請輸入分數...",
    lines=1)

intput_AI = gr.Textbox(
    label="對AI生成回答的意見",
    placeholder="您對AI回答的看法...",
    lines=3)


# Output textbox
output_result = gr.Textbox(
    label="AI的建議",
    placeholder="Output will appear here...",
    lines=10)

output_rel = gr.Textbox(
    label="AI回答的相關性得分",
    placeholder="AI回答是否與提問相關...",
    lines=1)

output_toxi = gr.Textbox(
    label="回答內容的危害性",
    placeholder="AI回答是否還有不當的言詞...",
    lines=1)


## Block 的寫法
with gr.Blocks() as demo:
    gr.Markdown("""## 「預立醫療照護諮商」( ACP)
                於下方欄位輸入輸入您想了解的事項, AI將提供您諮詢服務
    """)
    gr.themes.Default(primary_hue="red", secondary_hue="pink")

    with gr.Row():

        with gr.Column():
            input_box.render()
            btn = gr.Button(value="請按此鍵諮詢",variant="primary", scale=1)
            btn.click(fn=ACP_consultant,inputs=[input_box],outputs=[output_result, output_rel, output_toxi] )
            #gr.ClearButton()  ## check 功能, 似乎沒作用
        output_result.render()
    #input_Score.render()
    #intput_AI.render()
    #btn_1 = gr.Button(value="請按下對AI回答的評分及建議",variant="primary", scale=1)
    #btn_1.click(fn=score, inputs=[input_Score, intput_AI],)
    
    gr.Markdown("""## AI 的自我評分: """)
    with gr.Row():

        with gr.Column():
            output_rel.render()
        output_toxi.render()

demo.launch(share=False)
gr.close_all()
#demo.launch(share=True, server_port=int(os.environ['PORT3']))
#demo.queue().launch(share=True, server_port=int(os.environ['PORT4']))    

Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.


IMPORTANT: You are using gradio version 4.26.0, however version 4.29.0 is available, please upgrade.
--------


In [ ]:
## check result
import pandas as pd
pd.set_option('display.max_colwidth', None)

QA_record = pd.read_csv("QA_record.csv")

QA_record

,Question,Answer,Relevancy,Toxicity,Score,Suggestion,Relevance
0,什麼是「預立醫療照護諮商」( ACP),「預立醫療照護諮商」(ACP)是一個讓預立醫療決定書生效的法律程序。這是一個意願人（想簽署預立醫療決定書的人）與ACP諮商團隊、親屬或其他相關人進行溝通的過程。在這個過程中，主要討論的是當因重大意外或疾病使自己處於特定臨床條件後，想要接受或拒絕的維持生命治療和人工營養及流體餵養的醫療選擇。,0.867607,0.001430,85.0,還可以,NaN
1,任何人都可以進行「預立醫療照護諮商」 及「預立醫療決定」嗎?,滿 20歲具有完全行為能力者皆可進行「預立醫療照護諮商」及「預立醫療決定」。,0.875553,0.001453,90.0,可以,NaN
2,"不開心, 看人就打, 或是打傷自己, 那個男生太醜",這段描述可能暗示到一些情緒和行為問題，建議您尋求專業的心理諮詢或心理健康專家的幫助。如果您或他人有受傷的情況，也應該立即尋求醫療協助。,0.532915,0.001307,90.0,ok,NaN
3,根據「病人自主權利法」，預立醫療照護諮商門診是為了什麼目的？,根據「病人自主權利法」，預立醫療照護諮商門診的目的是讓意願人與諮商團隊、親屬或其他相關人進行溝通，主要是為了在特定臨床條件下，如重大意外或疾病發生時，確保病人能夠表達自己的意願，包括接受或拒絕哪些醫療措施。这个过程是为了尊重病人的醫療自主權，保障病人的善終權益。,0.824861,0.001489,90.0,出現幾個簡體字歐,NaN
4,誰可以申請預立醫療照護諮商,任何人都可以申請預立醫療照護諮商，無論是個人本身或是其親友、醫療委任代理人等相關人士。,0.684496,0.001638,60.0,NaN,NaN
...,...,...,...,...,...,...,...
70,我該如何修改預立醫療意願?,根據提供的資訊，您可以透過預立醫療照護諮商服務來修改您的預立醫療意願。您可以點選相關連結進行預約，然後在諮商過程中告知醫療專業人員您希望修改您的預立醫療意願。希望這能幫助到您。,NaN,0.001301,NaN,NaN,0.807419
71,什麼是醫療倫理?,醫療倫理是指在醫療實踐中所涉及的道德原則和價值觀，包括尊重病人的自主權、公平正義、不傷害病人等。這些原則和價值觀指導著醫療專業人員在與病人互動和治療過程中應該如何行事。,NaN,0.001670,NaN,NaN,0.368573
72,什麼是善終?,善終是指在生命接近終結時，能夠以尊嚴、舒適、無痛苦地離開人世，同時能夠與家人、醫護人員進行溝通和準備，讓生命的結束變得更有意義和平靜。在這個過程中，個人可以做出自己的醫療決定，包括選擇安寧緩和醫療等。,NaN,0.001437,NaN,NaN,0.409926
73,什麼是預立醫療照護諮商?,預立醫療照護諮商（ACP）是一種程序，讓個人在健康狀況良好時，與醫療團隊和家人討論未來可能需要的醫療照護選擇，包括對於維持生命治療和人工營養的意願。透過ACP，個人可以制定預立醫療決定書，明確表達自己的意願，以便在未來無法表達意願時，醫療團隊和家人可以依照其意願做出決定。,NaN,0.001512,NaN,NaN,0.733301
